# Neural-Network Fundamentals

**Workshop — Day 1**  
**Instructor:** Efraín Padilla — DLR IMF-ASP

Estimated duration: approximately two hours.

This notebook introduces optimization and the basic building blocks of neural networks through regression and classification examples motivated by remote sensing.

## Learning objectives

By the end of this session, participants should be able to:

- formulate a simple optimization problem;
- relate linear regression and least squares to a trainable model;
- explain and implement gradient descent;
- distinguish a loss function, a neuron, and an activation function;
- identify the difference between regression and classification; and
- interpret simple neural-network results for remote-sensing data.

## 0. Notation and setup

Suggested time: 5 minutes.



### How to read the notation in Day 1

We will use the same symbols throughout the notebook. Before doing any derivation, identify whether a symbol represents one number, a list of numbers, or a rectangular array of numbers.

| Symbol | Meaning | Typical shape |
|---|---|---|
| $N$ | number of training examples | one positive integer |
| $d$ | number of input features for one example | one positive integer |
| $q$ | number of output values; here usually $q=1$ | one positive integer |
| $i$ | index selecting one example, from $1$ to $N$ | integer |
| $x_i$ | input features for example $i$ | vector in $\mathbb{R}^{d}$ |
| $y_i$ | observed target for example $i$ | scalar in regression/classification here |
| $\hat y_i$ | model prediction for example $i$; the hat means “predicted” | scalar |
| $X$ | design matrix containing all input rows | $N\times d$ (or $N\times(d+1)$ with bias) |
| $y$ | column containing all observed targets | $N$-vector |
| $w$ | trainable feature weights | $d$-vector (or augmented vector) |
| $b$ | trainable intercept, also called bias | scalar |
| $\theta$ | generic name for all trainable parameters | vector or collection of arrays |
| $\mathcal{L}$ | average training loss, also called the objective | scalar |
| $\ell$ | loss for one example | scalar |
| $\eta$ | learning rate; the positive step-size chosen by us | scalar |
| $k$ | iteration number | integer |

A superscript $T$ means transpose, so $w^Tx$ is an inner product. The notation $\|v\|_2$ means the Euclidean length of vector $v$. A gradient, such as $\nabla_w\mathcal{L}$, is a vector containing one partial derivative for each component of $w$. The notation $\operatorname*{arg\,min}$ means “the parameter value at which the smallest objective is attained.”

In code, the correspondence is direct: `N` is the number of rows, `X` stores the rows, `y` stores the targets, `w` stores the parameters, `mse_half` evaluates $\mathcal{L}$, and `mse_gradient` evaluates $\nabla_w\mathcal{L}$.

## General description and introductory mathematics

A dataset contains input observations and corresponding targets. A model maps each input to a prediction, and training chooses parameters that make the average prediction error small:

$$\mathcal{L}(\theta)=\frac{1}{N}\sum_{i=1}^{N}\ell(f(x_i;\theta),y_i),\qquad \theta^\star=\operatorname*{arg\,min}_{\theta}\mathcal{L}(\theta).$$

The key pattern is simple: define a loss, identify the parameters that can change, and find the parameter values that make the loss as small as possible. We will return to this pattern throughout the notebook.

In [1]:
# --- Setup --- 

# Restart the kernel before running this cell if Matplotlib was already loaded.
import ipympl
# Enable interactive rotation and zooming for Matplotlib figures in VS Code.
get_ipython().run_line_magic("matplotlib", "ipympl")

# Import the numerical and plotting libraries used throughout Day 1.
import numpy as np
import matplotlib.pyplot as plt

# Choose plotting defaults that keep axes and mathematical curves readable.
plt.rcParams.update({"figure.figsize": (7, 4.5), "axes.grid": True, "font.size": 11})

## 1. Fundamentals of optimization

Suggested time: 30 minutes.



### 1.1 General description and introductory mathematics

For one example, an affine model combines features with weights and an intercept:

$$\hat y_i=w^Tx_i+b,\qquad \mathcal{L}(w)=\frac{1}{2N}\|Xw-y\|_2^2.$$

These equations introduce the general pattern. The derivation below makes each symbol and operation explicit.

First fix one training example. The vector $x_i$ is what the model receives, and the scalar $y_i$ is the answer we want the model to reproduce. The index $i$ tells us which example we mean; it is not a model parameter. The index runs from $1$ through $N$, where $N$ is the total number of examples.



### Definition

Let the data set be

$$
\mathcal{D}=\{(x_i,y_i)\}_{i=1}^{N},
\qquad x_i\in\mathbb{R}^{d},\quad y_i\in\mathbb{R}^{q}.
$$

Here $\mathcal{D}$ is the entire data set, $x_i$ is the $i$-th input vector, and $y_i$ is its observed target. The notation $x_i\in\mathbb{R}^{d}$ says that each input has $d$ numerical features. The notation $y_i\in\mathbb{R}^{q}$ allows $q$ output values; in most of this notebook $q=1$.

A model is a parameterized map. The symbol $\theta$ collects every number that training is allowed to change. For the simplest linear model, $\theta$ will be the weights and bias; we use $\theta$ here because the same idea will later describe an entire neural network.

$$
\hat y_i=f(x_i;\theta).
$$

The sample loss $\ell(\hat y_i,y_i)$ measures disagreement for one example. The first argument, $\hat y_i$, is the prediction; the second argument, $y_i$, is the observed answer. The empirical loss averages this disagreement over all $N$ examples:

$$
\mathcal{L}(\theta)=\frac{1}{N}\sum_{i=1}^{N}
\ell\left(f(x_i;\theta),y_i\right).
$$

Learning means selecting parameters that make the single scalar objective $\mathcal{L}(\theta)$ small:

$$
\theta^\star=\operatorname*{arg\,min}_{\theta}\mathcal{L}(\theta).
$$

This formulation does not mention neural networks. It applies to linear regression, logistic regression, multilayer perceptrons, and convolutional networks. What changes is the parameterized function, the loss, or the method used to evaluate its gradient.

#### A warm-up optimization problem: designing an aluminium can

Before optimizing a remote-sensing model, consider a familiar design problem. Suppose we want to manufacture a closed cylindrical can that holds a fixed volume $V$ while using as little aluminium as possible. Let $r>0$ be the radius and $h>0$ be the height. The quantity to minimize is the can's surface area:

$$
S(r,h)=2\pi r^2+2\pi rh.
$$

#### Illustration of the aluminium-can optimization problem
<div>
<img src="../assets/aluminium_can_optimization.png" width="500"/>
</div>

*A closed cylindrical can with radius $r$, height $h$, fixed volume $V$, and surface area $S$ to minimize.*

#### Deriving the optimal dimensions

Suggested time: 10 minutes.

The first term is the area of the two circular ends, and the second term is the area of the curved side. The volume constraint is

$$
V=\pi r^2h.
$$

We can use the constraint to eliminate the height:

$$
h(r)=\frac{V}{\pi r^2},
\qquad
S(r)=2\pi r^2+\frac{2V}{r}.
$$

The original two-variable design problem has now become a one-variable optimization problem. To differentiate it, use the power rule $d(r^2)/dr=2r$ and the equivalent form $1/r=r^{-1}$, whose derivative is $-r^{-2}$:

$$
\frac{dS}{dr}
=\frac{d}{dr}\left(2\pi r^2+2Vr^{-1}\right)
=2\pi(2r)+2V(-r^{-2})
=4\pi r-\frac{2V}{r^2}.
$$

At an interior minimum, this first derivative must be zero. Therefore

$$
4\pi r-\frac{2V}{r^2}=0
\quad\Longrightarrow\quad
4\pi r=\frac{2V}{r^2}
\quad\Longrightarrow\quad
4\pi r^3=2V
\quad\Longrightarrow\quad
r^3=\frac{V}{2\pi}.
$$

Therefore

$$
r^\star=\left(\frac{V}{2\pi}\right)^{1/3},
\qquad
h^\star=2r^\star.
$$

Why does the optimum satisfy $h^\star=2r^\star$? This relation is a consequence of the constraint and the first-order optimality condition; it is not an additional assumption. At the optimum,

$$
4\pi(r^\star)^3=2V
\quad\Longrightarrow\quad
V=2\pi(r^\star)^3.
$$

Substituting this result into the volume constraint gives

$$
h^\star=\frac{V}{\pi(r^\star)^2}
=\frac{2\pi(r^\star)^3}{\pi(r^\star)^2}
=2r^\star.
$$

Thus, once the optimal radius is known, the fixed-volume constraint determines the corresponding height, which is exactly twice the radius.

Differentiating once more gives

$$
\frac{d^2S}{dr^2}=4\pi+4Vr^{-3}=4\pi+\frac{4V}{r^3}>0
\qquad (r>0).
$$

The second derivative is positive for every $r>0$, so the objective is strictly convex and this stationary point is the unique minimum. The lesson is the same one we will use for machine learning: define a quantity to minimize, identify the parameters that can change, compute how the objective changes with those parameters, and verify that the result is a minimum.

#### Demonstration: explore the can objective

Suggested time: 10 minutes.

Change the volume below and observe how the optimal radius, height, and material requirement change. This one-dimensional objective is easier to inspect than a neural-network objective. Later, the scalar design variable $r$ will be replaced by a vector of trainable model parameters $\theta$.

In [ ]:
# Choose a can volume in cubic centimetres and evaluate the material objective.
can_volume = 330.0
radius_candidates = np.linspace(1.0, 8.0, 500)
surface_area = 2.0 * np.pi * radius_candidates**2 + 2.0 * can_volume / radius_candidates
optimal_radius = (can_volume / (2.0 * np.pi)) ** (1.0 / 3.0)
optimal_height = 2.0 * optimal_radius

# Plot the one-dimensional objective and mark the analytic minimizer.
plt.figure(figsize=(7, 4.5))
plt.plot(radius_candidates, surface_area, label="$S(r)$")
plt.scatter([optimal_radius], [2.0 * np.pi * optimal_radius**2 + 2.0 * can_volume / optimal_radius],
            color="tab:red", zorder=3, label="$r^\\star$")
plt.xlabel("can radius $r$ [cm]")
plt.ylabel("surface area $S(r)$ [cm$^2$]")
plt.title(f"Fixed volume {can_volume:.0f} cm$^3$: a simple optimization problem")
plt.legend()
plt.show()
print(f"optimal radius = {optimal_radius:.3f} cm")
print(f"optimal height = {optimal_height:.3f} cm")

## 2. Linear regression and least squares

Suggested time: 20 minutes.



### Why linear regression?

Linear regression is one of the simplest ways to predict a continuous quantity from one or more input features. It is useful as a first model because its parameters are easy to interpret, its objective has a clear mathematical form, and it provides a baseline against which more flexible models can be compared.

In remote sensing, one sample might be a pixel, an image patch, or an observation from a time series. The input features could be spectral bands, band ratios, vegetation indices, texture measures, topographic variables, or temporal summaries. The target could be a continuous quantity such as leaf-area index, biomass, soil moisture, surface temperature, or a laboratory measurement.

A linear model assumes that the target can be approximated by a weighted sum of the input features. This assumption is often too simple for a final remote-sensing model, but it is valuable for understanding the data, checking preprocessing, and establishing a transparent baseline. Spatial context and nonlinear interactions are not learned automatically; they must be added explicitly or handled by a more expressive model.



### 2.1 A synthetic regression problem

We first use one input feature so that the relationship can be plotted directly. The data will contain an underlying linear trend plus noise, which represents variability not explained by the feature.

The first visualization should show the observations, the trend we want to estimate, and the difference between a prediction and its observed target.



### 2.2 Mathematical formulation

Let the training dataset be

$$
\mathcal{D}=\{(x_i,y_i)\}_{i=1}^{N},\qquad x_i\in\mathbb{R}^{d},\quad y_i\in\mathbb{R}.
$$

Here, $N$ is the number of observations, $d$ is the number of features, $x_i$ is the feature vector for observation $i$, and $y_i$ is its measured target. The linear model predicts

$$
\hat{y}_i=w^Tx_i+b,
$$

where $w\in\mathbb{R}^{d}$ contains the feature weights and $b$ is the intercept or bias. The residual for observation $i$ is the prediction error

$$
e_i=\hat{y}_i-y_i.
$$

Least squares chooses $w$ and $b$ so that the sum of squared residuals is as small as possible. We use one half in the definition because it simplifies derivatives later:

$$
\mathcal{L}(w,b)=\frac{1}{2N}\sum_{i=1}^{N}(\hat{y}_i-y_i)^2.
$$

The square makes positive and negative errors contribute equally and penalizes large errors more strongly. The factor $1/N$ turns the sum into an average, so the loss is not directly determined by the number of observations.



### 2.3 Matrix form and the least-squares solution

It is convenient to include the bias in an augmented feature vector:

$$
\tilde{x}_i=\begin{bmatrix}1\\x_i\end{bmatrix},\qquad \theta=\begin{bmatrix}b\\w\end{bmatrix}.
$$

With the augmented design matrix $\tilde{X}$ and target vector $y$, all predictions can be written as $\hat{y}=\tilde{X}\theta$. The objective becomes

$$
\mathcal{L}(\theta)=\frac{1}{2N}\|\tilde{X}\theta-y\|_2^2.
$$

The key step is to set the gradient with respect to the parameters to zero. This gives the normal equations:

$$
\nabla_\theta\mathcal{L}(\theta)=\frac{1}{N}\tilde{X}^T(\tilde{X}\theta-y)=0\quad\Longrightarrow\quad\tilde{X}^T\tilde{X}\theta=\tilde{X}^Ty.
$$

The equation has a useful geometric interpretation. The residual vector $e=\tilde{X}\theta-y$ must be orthogonal to every column of $\tilde{X}$. If it had a component in one of those column directions, changing the corresponding parameter could reduce the squared error. The transpose $\tilde{X}^T$ collects these residual--feature correlations, so the gradient is zero exactly when all of them vanish.

Full column rank adds the condition needed to solve these equations uniquely. It makes the columns of $\tilde{X}$ linearly independent, so $\tilde{X}^T\tilde{X}$ is invertible. Because the least-squares loss is a convex quadratic, this unique stationary point is the global minimum:

$$
\theta^\star=(\tilde{X}^T\tilde{X})^{-1}\tilde{X}^Ty.
$$

Thus, setting the gradient to zero gives the normal equations; full column rank lets us turn those equations into a unique closed-form parameter vector. If the matrix is rank-deficient, the normal equations still describe least-squares minimizers, but the inverse does not exist and different parameter vectors may produce the same predictions.

In practical code, a least-squares solver is preferred to explicitly computing the inverse. The closed-form solution is useful here because it gives us a reference solution that we can later compare with gradient descent.

> **Workshop focus.** The essential ideas are the model, the residuals, and the loss. The normal-equation derivation is included as a reference; if time is limited, it can be summarized briefly before moving to gradient descent, which is the more direct bridge to neural-network training.



### 2.4 Optional reference: deriving the least-squares solution step by step

> This subsection is provided for later consultation. It is not necessary to cover every step during the live workshop.



#### Step 1: include the intercept in the parameter vector

The model $\hat{y}_i=w^Tx_i+b$ has a special parameter, $b$, that is not multiplied by an input feature. We can treat the intercept like any other weight by adding a constant feature equal to one:

$$
\tilde{x}_i=\begin{bmatrix}1\\x_i\end{bmatrix},\qquad \theta=\begin{bmatrix}b\\w\end{bmatrix}.
$$

The prediction is now $\hat{y}_i=\tilde{x}_i^T\theta$. If the original input has $d$ features, then $\tilde{x}_i$ has $d+1$ entries.



#### Step 2: stack all observations into a design matrix

Place one augmented input vector in each row of the design matrix:

$$
\tilde{X}=\begin{bmatrix}
1 & x_1^T\\
1 & x_2^T\\
\vdots & \vdots\\
1 & x_N^T
\end{bmatrix}\in\mathbb{R}^{N\times(d+1)}.
$$

The target vector is $y=[y_1,\ldots,y_N]^T$. Multiplying the design matrix by $\theta$ produces all predictions at once:

$$
\hat{y}=\tilde{X}\theta.
$$



#### Step 3: write the loss using a vector norm

The residual vector is $e=\hat{y}-y=\tilde{X}\theta-y$. Its squared Euclidean norm is the sum of the squared residuals, so

$$
\mathcal{L}(\theta)=\frac{1}{2N}\|\tilde{X}\theta-y\|_2^2.
$$

Using $\|a\|_2^2=a^Ta$, the same objective can be written as

$$
\mathcal{L}(\theta)=\frac{1}{2N}(\tilde{X}\theta-y)^T(\tilde{X}\theta-y).
$$



#### Step 4: expand the quadratic expression

Expanding the product gives

$$
\mathcal{L}(\theta)=\frac{1}{2N}\left(\theta^T\tilde{X}^T\tilde{X}\theta-2y^T\tilde{X}\theta+y^Ty\right).
$$

The final term, $y^Ty$, does not depend on $\theta$. It affects the value of the loss but not the location of its minimum.



#### Step 5: differentiate with respect to the parameters

The gradient of the loss is

$$
\nabla_\theta\mathcal{L}(\theta)=\frac{1}{N}\tilde{X}^T(\tilde{X}\theta-y).
$$

The transpose $\tilde{X}^T$ maps the residuals from observation space back to parameter space. It combines the residuals according to the feature values.



#### Step 6: impose the stationary-point condition

At a differentiable interior minimum, the gradient is zero:

$$
\frac{1}{N}\tilde{X}^T(\tilde{X}\theta-y)=0.
$$

Multiplying by $N$ and rearranging gives the normal equations:

$$
\tilde{X}^T\tilde{X}\theta=\tilde{X}^Ty.
$$



#### Step 7: understand the full-column-rank condition

The design matrix $\tilde{X}$ has full column rank when its $d+1$ columns are linearly independent:

$$
\operatorname{rank}(\tilde{X})=d+1.
$$

This requires at least $N\ge d+1$ observations and no exact linear dependence among the columns. Under this condition, $\tilde{X}^T\tilde{X}$ is invertible. To see why, take any nonzero vector $v$:

$$
v^T\tilde{X}^T\tilde{X}v=\|\tilde{X}v\|_2^2>0.
$$

The strict inequality follows because full column rank implies $\tilde{X}v\ne0$ whenever $v\ne0$. Therefore $\tilde{X}^T\tilde{X}$ is positive definite and has an inverse.



#### Step 8: solve the normal equations

Multiplying the normal equations by $(\tilde{X}^T\tilde{X})^{-1}$ gives

$$
\theta^\star=(\tilde{X}^T\tilde{X})^{-1}\tilde{X}^Ty.
$$

This is the closed-form least-squares solution. In numerical work, use a QR- or SVD-based least-squares solver instead of explicitly forming the inverse. This is especially important in remote sensing, where spectral bands and derived indices can be highly correlated.



#### A small hand-worked example

For one feature and three observations, let $x=[1,2,3]^T$ and $y=[2,3,5]^T$. Including the intercept gives

$$
\tilde{X}=\begin{bmatrix}1&1\\1&2\\1&3\end{bmatrix},\qquad \tilde{X}^T\tilde{X}=\begin{bmatrix}3&6\\6&14\end{bmatrix},\qquad \tilde{X}^Ty=\begin{bmatrix}10\\23\end{bmatrix}.
$$

The determinant of $\tilde{X}^T\tilde{X}$ is $3\cdot14-6\cdot6=6$, so the matrix is invertible. Solving the normal equations gives $\theta^\star=[1/3,3/2]^T$. Thus the fitted model is $\hat{y}=1/3+(3/2)x$.

If two columns were duplicates or exact linear combinations, the determinant would be zero and the inverse would not exist. A least-squares solver or pseudoinverse can still provide a solution, but the parameters may no longer be unique.



### 2.5 Visual interpretation

The fitted line or plane is the set of predictions produced by the optimal parameters. We will first use one feature, where the model is a line, and then use two features, where the model becomes a plane. In both cases, the plot will make the parameters visible: the bias controls the offset and each weight controls the contribution of its corresponding feature.

#### 2.5.1 One input feature: a fitted line

For one feature, $\hat{y}=b+wx$. Here, $w$ is the weight of the input feature; in one dimension it is the slope of the line. The parameter $b$ is the bias or intercept, so it determines the predicted value when $x=0$.

The data-generating process is

$$
x_i\sim\mathcal{U}(0,10),\qquad y_i=1.5+2.2x_i+\varepsilon_i,\qquad \varepsilon_i\sim\mathcal{N}(0,2^2).
$$

The feature $x$ is sampled from a uniform distribution between 0 and 10. Its theoretical mean is $5$ and its theoretical standard deviation is $(10-0)/\sqrt{12}\approx2.89$. The noise $\varepsilon$ is sampled from a normal distribution with mean $0$ and standard deviation $2$; therefore, `rng.normal(0, 2.0, n_samples)` uses $0$ as the mean and $2.0$ as the standard deviation.

Before noise is added, the true line has bias $b=1.5$ and weight $w=2.2$. The noise moves each observation above or below this line. Consequently, the sample mean and standard deviation will not be exactly equal to their theoretical values, and the fitted parameters will be close to, but not exactly equal to, $1.5$ and $2.2$. The random seed makes this particular dataset reproducible.

In [ ]:
rng = np.random.default_rng(7)
n_samples = 40
x = rng.uniform(0, 10, n_samples)
y = 1.5 + 2.2 * x + rng.normal(0, 2.0, n_samples)

# The augmented design matrix has one row per observation: [1, x_i].
# The column of ones is what allows the model to learn an intercept (bias).
X = np.c_[np.ones(n_samples), x]
# theta = [b, w]^T, so the vectorized predictions are y_hat = X @ theta.
# In other words, each row computes b + w * x_i.
theta, *_ = np.linalg.lstsq(X, y, rcond=None)
# least_squares minimizes ||X @ theta - y||_2^2 and returns the best parameters.
b, w = theta

x_plot = np.linspace(x.min(), x.max(), 200)
y_plot = b + w * x_plot
y_pred = b + w * x
equation = fr"$\hat{{y}} = {b:.2f} + ({w:.2f})x$"
parameters = f"$b = {b:.2f}$\n$w = {w:.2f}$"

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, color="tab:blue", alpha=0.8, label="observations")
ax.plot(x_plot, y_plot, color="tab:red", linewidth=2.5, label="least-squares model")
for xi, yi, ypi in zip(x, y, y_pred):
    ax.plot([xi, xi], [yi, ypi], color="0.65", linewidth=0.8, alpha=0.6)

ax.text(0.04, 0.96, equation + "\n" + parameters, transform=ax.transAxes,
        va="top", bbox=dict(boxstyle="round", facecolor="white", alpha=0.9))
ax.set_title("One-feature linear regression")
ax.set_xlabel("Feature $x$")
ax.set_ylabel("Target $y$")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

#### 2.5.2 Two input features: a fitted plane

With two features, $\hat{y}=b+w_1x_1+w_2x_2$. The same least-squares procedure now fits a plane in the three-dimensional space $(x_1,x_2,y)$. The plane is the two-feature analogue of the line above.

The data are generated as

$$
x_{1i}\sim\mathcal{U}(-3,3),\qquad x_{2i}\sim\mathcal{U}(-2,2),
$$
$$
y_i=4.0+1.4x_{1i}-2.1x_{2i}+\varepsilon_i,\qquad \varepsilon_i\sim\mathcal{N}(0,1.2^2).
$$

Both features have theoretical mean $0$. Their theoretical standard deviations are $6/\sqrt{12}\approx1.73$ for $x_1$ and $4/\sqrt{12}\approx1.15$ for $x_2$. The noise has mean $0$ and standard deviation $1.2$. Thus, the true plane has bias $b=4.0$, weights $w_1=1.4$ and $w_2=-2.1$, while the fitted values vary slightly because of the finite sample and the added noise.

> The setup cell enables Matplotlib's `ipympl` backend. In VS Code, restart the kernel, run the setup cell first, and then run this visualization cell. Interact with the 3D figure by dragging to rotate and using the scroll wheel to zoom.

In [ ]:
rng = np.random.default_rng(21)
n_samples = 80
x1 = rng.uniform(-3, 3, n_samples)
x2 = rng.uniform(-2, 2, n_samples)
y = 4.0 + 1.4 * x1 - 2.1 * x2 + rng.normal(0, 1.2, n_samples)

# Each row of the augmented design matrix is [1, x_1i, x_2i].
# The first column represents the bias; the other columns represent the two features.
X = np.c_[np.ones(n_samples), x1, x2]
# theta = [b, w_1, w_2]^T, hence y_hat = X @ theta.
# Every row evaluates b + w_1*x_1i + w_2*x_2i.
theta, *_ = np.linalg.lstsq(X, y, rcond=None)
# This minimizes the total squared residual ||X @ theta - y||_2^2.
b, w1, w2 = theta

x1_grid = np.linspace(x1.min(), x1.max(), 30)
x2_grid = np.linspace(x2.min(), x2.max(), 30)
X1, X2 = np.meshgrid(x1_grid, x2_grid)
Y_plane = b + w1 * X1 + w2 * X2
equation = fr"$\hat{{y}} = {b:.2f} + ({w1:.2f})x_1 + ({w2:.2f})x_2$"
parameters = f"$b = {b:.2f}$\n$w_1 = {w1:.2f}$\n$w_2 = {w2:.2f}$"

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(x1, x2, y, color="tab:blue", alpha=0.8, label="observations")
ax.plot_surface(X1, X2, Y_plane, color="tab:red", alpha=0.45,
                edgecolor="none", label="least-squares plane")

ax.text2D(0.03, 0.96, equation + "\n" + parameters, transform=ax.transAxes,
          va="top", bbox=dict(boxstyle="round", facecolor="white", alpha=0.9))
ax.set_title("Two-feature linear regression")
ax.set_xlabel("Feature $x_1$", labelpad=8)
ax.set_ylabel("Feature $x_2$", labelpad=8)
ax.set_zlabel("Target $y$", labelpad=8)
ax.set_box_aspect((1.2, 1.0, 0.9))
ax.view_init(elev=10, azim=10)
plt.show()

## 3. Gradient descent

Suggested time: 20 minutes.



### 3.1 Why gradient descent?

Gradient descent is an iterative optimization algorithm. Instead of solving the normal equations in one large linear-algebra operation, it starts from an initial parameter vector and repeatedly moves in the direction that decreases the loss. This is the basic optimization mechanism used to train neural networks.

For $N$ observations and $d$ features, the normal-equation approach requires forming $\tilde{X}^T\tilde{X}$, which costs approximately $O(Nd^2)$ operations, and solving a $(d+1)\times(d+1)$ system, which costs approximately $O(d^3)$. A QR- or SVD-based solver is numerically safer, but it still becomes expensive when the number of features is large.

A full-batch gradient evaluation computes $\tilde{X}^T(\tilde{X}\theta-y)$ and costs approximately $O(Nd)$ per iteration. After $T$ iterations, the total cost is $O(TNd)$, and mini-batch methods can reduce the cost and memory used by each individual update. Gradient descent is therefore especially useful when the dataset is too large to fit comfortably in memory, when features arrive in batches, or when the model is nonlinear and no closed-form solution exists.

The trade-off is that gradient descent may require many iterations and depends on the learning rate and conditioning of the problem. It does not automatically make every small problem faster; its advantage is scalability and its direct connection to neural-network training. In remote sensing, this matters because a dataset can contain millions of pixels or patches, many spectral and spatial features, and models with millions of trainable parameters.



### 3.2 The gradient and update rule

For the linear regression loss

$$
\mathcal{L}(\theta)=\frac{1}{2N}\|\tilde{X}\theta-y\|_2^2,
$$

the gradient is

$$
\nabla_\theta\mathcal{L}(\theta)=\frac{1}{N}\tilde{X}^T(\tilde{X}\theta-y).
$$

The negative gradient points in the direction of steepest local decrease of the loss. With learning rate $\alpha>0$, gradient descent updates the parameters according to

$$
\theta_{k+1}=\theta_k-\alpha\nabla_\theta\mathcal{L}(\theta_k).
$$

The subscript $k$ denotes the iteration. Each update uses the current residuals to decide how the weights and bias should change. In a neural network, the same idea is applied to all trainable parameters, with the gradient obtained through backpropagation.



### 3.3 Learning rate and convergence

The learning rate controls the step size. If it is too small, the iterates move toward the minimum slowly. If it is too large, they can oscillate around the minimum or diverge. For a simple one-dimensional function, we can see the entire process directly.

Consider $f(z)=\frac{1}{2}(z-2)^2$. Its derivative is $f'(z)=z-2$, so the unique minimum is at $z^\star=2$. Gradient descent becomes

$$
z_{k+1}=z_k-\alpha(z_k-2).
$$

The next cell plots the function, the successive iterates, and the loss value at every step. The arrows show the direction selected by the negative gradient.

In [5]:
# The function and derivative have a known minimum at z_star = 2.
def f(z):
    return 0.5 * (z - 2.0) ** 2

def grad_f(z):
    return z - 2.0

z = -4.0
learning_rate = 0.25
n_steps = 12
z_history = [z]
f_history = [f(z)]

for _ in range(n_steps):
    # This is the gradient-descent update z_new = z - alpha * grad_f(z).
    z = z - learning_rate * grad_f(z)
    z_history.append(z)
    f_history.append(f(z))

z_grid = np.linspace(-5, 5, 400)
fig, (ax_function, ax_loss) = plt.subplots(1, 2, figsize=(12, 4.5))

ax_function.plot(z_grid, f(z_grid), color="tab:blue", linewidth=2, label="f(z)")
ax_function.scatter(z_history, f_history, c=np.arange(len(z_history)), cmap="viridis", zorder=3)
for k in range(len(z_history) - 1):
    ax_function.annotate("", xy=(z_history[k + 1], f_history[k + 1]),
                        xytext=(z_history[k], f_history[k]),
                        arrowprops=dict(arrowstyle="->", color="0.35", lw=1))
ax_function.axvline(2.0, color="tab:red", linestyle="--", label="minimum $z^\star=2$")
ax_function.set_title("Iterates on the function")
ax_function.set_xlabel("Parameter $z$")
ax_function.set_ylabel("Function value $f(z)$")
ax_function.legend()

ax_loss.plot(range(len(f_history)), f_history, marker="o", color="tab:purple")
ax_loss.set_title("Loss decreases at each iteration")
ax_loss.set_xlabel("Iteration $k$")
ax_loss.set_ylabel("$f(z_k)$")
ax_loss.grid(alpha=0.25)
fig.tight_layout()
plt.show()

### 3.4 Gradient descent on the regression problem

The same iterative rule can be applied to the two parameters of a one-feature regression model, $\theta=[b,w]^T$. The left panel below shows the loss surface in parameter space. The gradient-descent path moves along this surface toward the minimum, marked by the analytical least-squares solution. The right panel shows the loss decreasing with iterations.

In [ ]:
rng = np.random.default_rng(11)
n_samples = 40
x_reg = rng.uniform(0, 10, n_samples)
y_reg = 1.5 + 2.2 * x_reg + rng.normal(0, 1.5, n_samples)

# The augmented design matrix represents y_hat = X @ theta, with theta = [b, w]^T.
X_reg = np.c_[np.ones(n_samples), x_reg]

# This is the analytical reference solution from the least-squares problem.
theta_star, *_ = np.linalg.lstsq(X_reg, y_reg, rcond=None)

# Start from an initial parameter vector and repeatedly follow the negative gradient.
theta = np.array([0.0, 0.0])
learning_rate = 0.02
n_steps = 1000
theta_history = [theta.copy()]
loss_history = [0.5 * np.mean((X_reg @ theta - y_reg) ** 2)]

for _ in range(n_steps):
    # The residual is X @ theta - y; multiplying by X.T gives the parameter gradient.
    gradient = X_reg.T @ (X_reg @ theta - y_reg) / n_samples
    theta = theta - learning_rate * gradient
    theta_history.append(theta.copy())
    loss_history.append(0.5 * np.mean((X_reg @ theta - y_reg) ** 2))

theta_history = np.array(theta_history)

# Evaluate the loss on a grid to visualize the two-dimensional parameter surface.
b_grid = np.linspace(-2, 5, 120)
w_grid = np.linspace(0.5, 3.2, 120)
B, W = np.meshgrid(b_grid, w_grid)
predictions = B[..., None] + W[..., None] * x_reg
loss_surface = 0.5 * np.mean((predictions - y_reg) ** 2, axis=2)
levels = np.geomspace(max(loss_surface.min(), 1e-3), loss_surface.max(), 18)

fig, (ax_path, ax_loss) = plt.subplots(1, 2, figsize=(12, 4.5))
ax_path.contour(B, W, loss_surface, levels=levels, cmap="viridis")
ax_path.plot(theta_history[:, 0], theta_history[:, 1], color="tab:red", linewidth=1.5)
ax_path.scatter(theta_history[::20, 0], theta_history[::20, 1], c=np.arange(0, len(theta_history), 20),
                cmap="plasma", s=18, zorder=3)
ax_path.scatter(theta_star[0], theta_star[1], color="black", marker="*", s=170,
                label="least-squares minimum", zorder=4)
ax_path.set_title("Gradient-descent path in parameter space")
ax_path.set_xlabel("Bias $b$")
ax_path.set_ylabel("Weight $w$")
ax_path.legend()

ax_loss.semilogy(loss_history, color="tab:purple", linewidth=2)
ax_loss.scatter([0, n_steps], [loss_history[0], loss_history[-1]], color="tab:red", zorder=3)
ax_loss.set_title("Loss convergence")
ax_loss.set_xlabel("Iteration $k$")
ax_loss.set_ylabel("$\mathcal{L}(\theta_k)$")
ax_loss.grid(alpha=0.25)
fig.tight_layout()
plt.show()

print(f"Least-squares solution: b = {theta_star[0]:.3f}, w = {theta_star[1]:.3f}")
print(f"Gradient-descent solution: b = {theta[0]:.3f}, w = {theta[1]:.3f}")

## 4. Loss functions and the neuron

Suggested time: 15 minutes.



### 4.1 From a linear model to a neuron

Relate weights, bias, weighted sum, output, and trainable parameters.



### 4.2 Loss functions

Introduce a suitable loss for regression and classification, and discuss what the loss measures.



### 4.3 Computational graph

Show the forward pass and the role of derivatives in updating the parameters.

In [6]:
# TODO: visualize a neuron and compare loss functions
pass

## 5. Regression and classification

Suggested time: 20 minutes.



### 5.1 Activation functions

Introduce the identity, sigmoid, and other relevant activation functions. Discuss their effect on the model output.



### 5.2 Regression

Train a simple model for a continuous target.



### 5.3 Classification

Train a simple model for a categorical target and visualize the decision boundary or class probabilities.

In [7]:
# TODO: compare activation functions and train simple regression/classification models
pass

## 6. Remote-sensing examples

Suggested time: 10 minutes.



### 6.1 Regression example

Define a small remote-sensing regression task, such as estimating a continuous biophysical variable from spectral features.



### 6.2 Classification example

Define a small remote-sensing classification task, such as assigning land-cover classes from spectral features.



### 6.3 Interpretation

Visualize predictions, errors, and class confusion. Discuss what the model does and does not learn from the inputs.

In [8]:
# TODO: add a small prepared remote-sensing dataset and the corresponding examples
pass

## 7. Summary and discussion

Suggested time: 5 minutes.

- Optimization turns model fitting into a parameter-search problem.
- Linear regression provides a simple bridge to trainable neurons.
- Loss functions, gradients, and activation functions determine how models learn.
- Remote-sensing applications require careful interpretation of inputs, targets, and errors.

**Discussion prompt:** What information is available in the input features, and what information is missing from the model?